In [13]:
from ucimlrepo import fetch_ucirepo
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [14]:
wine_quality = fetch_ucirepo(name='Wine Quality')
wine_data = wine_quality['data']['original']
wine_data

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,color
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white


In [16]:
seed = 42 #if it must be 42 :P

wine_X_train_val, wine_X_test, wine_y_train_val, wine_y_test = train_test_split(
    wine_data.drop(columns = ['quality']), wine_data['quality'], test_size=0.2, random_state=42
)

wine_X_tr, wine_X_val, wine_y_tr, wine_y_val = train_test_split(
    wine_X_train_val, wine_y_train_val, test_size=0.25, random_state=42
)
print(wine_X_tr.shape, wine_X_val.shape, wine_y_tr.shape, wine_y_val.shape)


(3897, 12) (1300, 12) (3897,) (1300,)


In [4]:
raw_features = wine_quality_data.drop(columns=['quality']).assign(
    color=wine_quality_data['color'].transform(lambda x: (x == 'red').astype(float))
).values

scaler = StandardScaler()
features = torch.tensor(scaler.fit_transform(raw_features), dtype=torch.float32)
num_features = features.shape[1]

min_quality = wine_quality_data['quality'].min()
num_classes = wine_quality_data['quality'].max() - min_quality + 1
labels = torch.tensor((wine_quality_data['quality'].values - min_quality), dtype=torch.long)

dataset = torch.utils.data.TensorDataset(features, labels)
trainloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)


In [5]:
class Net(nn.Module):
    def __init__(self, num_classes):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(num_features, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, input):
        f1 = F.relu(self.fc1(input))
        f2 = F.relu(self.fc2(self.dropout(f1)))
        f3 = F.relu(self.fc3(self.dropout(f2)))
        return self.fc4(f3)

net = Net(num_classes)


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)


In [7]:
for epoch in range(50):
    net.train()
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, batch_labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    net.eval()
    with torch.no_grad():
        correct = (torch.argmax(net(features), dim=1) == labels).sum().item()
        acc = correct / len(labels) * 100
    print(f'[Epoch {epoch + 1:2d}] loss: {running_loss / len(trainloader):.3f}  Accuracy: {acc:.2f}%')

print('Finished Training')


[Epoch  1] loss: 1.257  Accuracy: 52.66%
[Epoch  2] loss: 1.116  Accuracy: 55.70%
[Epoch  3] loss: 1.086  Accuracy: 55.29%
[Epoch  4] loss: 1.072  Accuracy: 55.49%
[Epoch  5] loss: 1.071  Accuracy: 55.59%
[Epoch  6] loss: 1.060  Accuracy: 56.96%
[Epoch  7] loss: 1.049  Accuracy: 56.98%
[Epoch  8] loss: 1.046  Accuracy: 57.26%
[Epoch  9] loss: 1.039  Accuracy: 57.10%
[Epoch 10] loss: 1.032  Accuracy: 57.83%
[Epoch 11] loss: 1.021  Accuracy: 58.23%
[Epoch 12] loss: 1.019  Accuracy: 58.37%
[Epoch 13] loss: 1.024  Accuracy: 57.86%
[Epoch 14] loss: 1.009  Accuracy: 58.15%
[Epoch 15] loss: 1.016  Accuracy: 57.95%
[Epoch 16] loss: 1.015  Accuracy: 59.01%
[Epoch 17] loss: 1.011  Accuracy: 58.86%
[Epoch 18] loss: 1.010  Accuracy: 58.53%
[Epoch 19] loss: 1.000  Accuracy: 58.97%
[Epoch 20] loss: 0.995  Accuracy: 58.55%
[Epoch 21] loss: 0.995  Accuracy: 59.29%
[Epoch 22] loss: 0.991  Accuracy: 59.40%
[Epoch 23] loss: 0.991  Accuracy: 59.41%
[Epoch 24] loss: 0.984  Accuracy: 59.75%
[Epoch 25] loss: